In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch

In [ ]:
# Load history

history_path = 'mhubert_base_swa_10min_history.json'

with open(history_path, 'r') as f:
    history = json.load(f)

train_loss = history['train_loss']
dev_loss = history['dev_loss']
epochs = np.arange(1, len(train_loss) + 1)

In [ ]:
# Figure 1: loss evolution

plt.figure(figsize=(12, 6))
plt.plot(epochs, train_loss, label='Train Loss', color='#1f77b4', lw=2, alpha=0.8)
plt.plot(epochs, dev_loss, label='Dev Loss', color='#ff7f0e', lw=2)

best_epoch = 60
best_loss = dev_loss[best_epoch - 1]

plt.scatter(best_epoch, best_loss, color='red', s=100, zorder=5, label=f'Best Model (Epoch {best_epoch})')
plt.annotate(f'Min Dev Loss: {best_loss:.4f}', 
             xy=(best_epoch, best_loss), 
             xytext=(best_epoch + 5, best_loss + 0.2),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8))

plt.title('Loss Evolution - ASR mHuBERT-147 Swahili', fontsize=14)
plt.xlabel('Number of epochs', fontsize=12)
plt.ylabel('CTC Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig('loss_curves_mhubert_swahili_10min.png', dpi=300)
plt.show()

In [ ]:
# Load model and weights

checkpoint = torch.load('mhubert_base_swa_10min_best.pt', map_location='cpu')
state_dict = checkpoint['model_state_dict']

weight_key = [k for k in state_dict.keys() if 'weights' in k][0]
raw_weights = state_dict[weight_key]

In [ ]:
# Figure 2: distribution of layer weights

weight_matrix = raw_weights.reshape(1, -1)

plt.figure(figsize=(12, 3))
sns.heatmap(
    weight_matrix, 
    annot=True,
    fmt=".2f", 
    cmap='magma',
    xticklabels=range(13),
    cbar_kws={'label': 'raw weights'}
)

plt.title("Distribution of Layer Weights - ASR mHuBERT-147 Swahili ", fontsize=14)
plt.xlabel("Layer index (0=CNN, 1-12=Transformer)")
plt.savefig('heatmap_weights_swahili.png', bbox_inches='tight')
plt.show()